# Speech Denoising — Notebook 01: EDA & Data Exploration

Dataset: VoiceBank+DEMAND  
Goal: Explore raw audio files, verify dataset integrity, visualise waveforms and spectrograms.

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import librosa as lb
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display
from tqdm import tqdm
from pesq import pesq

## Dataset Paths

In [ ]:
clean_trainset = Path("data/archive/clean_trainset_28spk_wav/")
noisy_trainset = Path("data/archive/noisy_trainset_28spk_wav/")
clean_testset  = Path("data/archive/clean_testset_wav/")
noisy_testset  = Path("data/archive/noisy_testset_wav/")

for p in [clean_trainset, noisy_trainset, clean_testset, noisy_testset]:
    print(f"{p}: {p.exists()}")

## Dataset Size

In [ ]:
clean_trainset_files = sorted(clean_trainset.glob("*.wav"))
noisy_trainset_files = sorted(noisy_trainset.glob("*.wav"))
clean_testset_files  = sorted(clean_testset.glob("*.wav"))
noisy_testset_files  = sorted(noisy_testset.glob("*.wav"))

print(f"Train clean: {len(clean_trainset_files)}")
print(f"Train noisy: {len(noisy_trainset_files)}")
print(f"Test  clean: {len(clean_testset_files)}")
print(f"Test  noisy: {len(noisy_testset_files)}")

## Load Sample Pair

In [ ]:
sample_name = "p226_004.wav"

audio_array_clean, sample_rate_clean = lb.load(clean_trainset / sample_name, sr=None)
audio_array_noisy, sample_rate_noisy = lb.load(noisy_trainset / sample_name, sr=None)

print(f"Sample rate: {sample_rate_clean} Hz")
print(f"Length:      {len(audio_array_clean)} samples ({len(audio_array_clean)/sample_rate_clean:.2f} sec)")

In [ ]:
print("Clean:")
display(Audio(audio_array_clean, rate=sample_rate_clean))
print("Noisy:")
display(Audio(audio_array_noisy, rate=sample_rate_noisy))

## Waveform Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(audio_array_clean)
axes[0].set_title("Clean")
axes[0].set_ylabel("Amplitude")

axes[1].plot(audio_array_noisy)
axes[1].set_title("Noisy")
axes[1].set_ylabel("Amplitude")
axes[1].set_xlabel("Samples")

plt.tight_layout()
plt.show()

## STFT Spectrogram Visualisation

In [ ]:
audio_array_clean_stft = lb.amplitude_to_db(np.abs(lb.stft(audio_array_clean)))
audio_array_noisy_stft = lb.amplitude_to_db(np.abs(lb.stft(audio_array_noisy)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lb.display.specshow(audio_array_clean_stft, sr=sample_rate_clean,
                    x_axis="time", y_axis="hz", ax=axes[0])
axes[0].set_title("Clean — STFT Spectrogram")

lb.display.specshow(audio_array_noisy_stft, sr=sample_rate_noisy,
                    x_axis="time", y_axis="hz", ax=axes[1])
axes[1].set_title("Noisy — STFT Spectrogram")

plt.tight_layout()
plt.show()